In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load data
df = pd.read_csv('data/user_nutritional_data.csv')  # your path

# 2. Create BMI if not there
df['BMI'] = df['Weight'] / ((df['Height']/100)**2)

# 3. Create Health Score (your original formula)
df['cal_norm'] = (df['Calories'] - df['Calories'].min()) / (df['Calories'].max() - df['Calories'].min())
df['carb_norm'] = (df['Carbs'] - df['Carbs'].min()) / (df['Carbs'].max() - df['Carbs'].min())
df['prot_norm'] = (df['Proteins'] - df['Proteins'].min()) / (df['Proteins'].max() - df['Proteins'].min())
df['fat_norm'] = (df['Fats'] - df['Fats'].min()) / (df['Fats'].max() - df['Fats'].min())
bmi_penalty = np.clip((df['BMI'] - 22) / 10, -1, 1)
df['Health_Score'] = np.clip(70 + 20*df['prot_norm'] - 10*df['fat_norm'] - 10*bmi_penalty, 0, 100)

# 4. Create Health Risk (BMI-based, balanced)
df['Health_Risk'] = np.where(df['BMI'] > 30, 'High', 'Low')

# 5. Feature engineering
df['carb_to_protein'] = df['Carbs'] / (df['Proteins'] + 1e-3)
df['fat_to_cal'] = df['Fats'] / (df['Calories'] + 1e-3)
df['meals_x_exercise'] = df['Daily meals frequency'] * df['Physical exercise']

# 6. Features + Split + Scale (same as Step 3)
feature_cols = [
    'Age','BMI','Calories','Carbs','Proteins','Fats',
    'Daily meals frequency','Physical exercise',
    'carb_to_protein','fat_to_cal','meals_x_exercise'
]
X = df[feature_cols]
y_reg = df['Health_Score']
y_cls = df['Health_Risk']

X_train, X_test, y_reg_train, y_reg_test, y_cls_train, y_cls_test = train_test_split(
    X, y_reg, y_cls, test_size=0.2, random_state=42, stratify=y_cls
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Split complete!")
print("Train shape:", X_train_scaled.shape)
print("Train classes:", pd.Series(y_cls_train).value_counts())
